<a href="https://colab.research.google.com/github/MarioDLCruz/PolisemiaV1/blob/main/demo_embeddings_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModel
import torch

In [2]:
import pandas as pd

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
def predict(text, bs=128, max_seq_length=128):
    embeddings = []
    all_tokens = {} # Store all tokens to return later
    for i in range(0, len(text), bs):
        if i//bs%100==99:
            print(i, "/", len(text))

        # Ensure consistent padding and truncation across all batches
        tokens = tokenizer(
            text[i: i+bs],
            return_tensors="pt",
            padding='max_length', # Use max_length padding
            truncation=True,
            max_length=max_seq_length # Use the specified max_seq_length
        )

        t = {"input_ids": tokens['input_ids'].to(device), "attention_mask": tokens['attention_mask'].to(device)}

        with torch.no_grad():
            pred = model(**t)

        emb = pred.hidden_states[-1]
        embeddings.append(emb.cpu())

        # Concatenate tokens['input_ids'] and tokens['attention_mask'] across batches
        # This is needed if you want to return all tokens as a single tensor
        if not all_tokens: # If all_tokens is empty, initialize it
            all_tokens['input_ids'] = tokens['input_ids'].cpu()
            all_tokens['attention_mask'] = tokens['attention_mask'].cpu()
        else: # Otherwise, concatenate
             all_tokens['input_ids'] = torch.cat((all_tokens['input_ids'], tokens['input_ids'].cpu()), dim=0)
             all_tokens['attention_mask'] = torch.cat((all_tokens['attention_mask'], tokens['attention_mask'].cpu()), dim=0)


    # Concatenate the embeddings from all batches along the batch dimension (dim=0)
    return torch.concatenate(embeddings, dim=0), all_tokens

In [5]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(device)
tokenizer = AutoTokenizer.from_pretrained("guillermoruiz/mex_state")
model = AutoModelForMaskedLM.from_pretrained("guillermoruiz/mex_state", output_hidden_states=True)
model = model.to(device)

cuda:0


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.27k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/650k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/165M [00:00<?, ?B/s]

In [ ]:
text = ["Querétaro _GEO _USR hermanaaaaa el disco, se viene otra biblia",
"Chihuahua _GEO Se me hace que a Lyn may le pusieron mucho botox en las cejas... _URL",
       "Puebla _GEO _USR 26 pte 26 de septiembre y 3 sur frente s la notaria publica #1"
      ]
emb = predict(text)

In [6]:
file_path = "/content/drive/MyDrive/diezmil.txt"
with open(file_path, "r", encoding="utf-8") as file:
    text = [line.strip() for line in file if line.strip()]
    emb, tokens = predict(text) # Update to unpack both embeddings and tokens
    print(emb.shape) # Print shape to verify

torch.Size([10000, 128, 512])


In [ ]:
emb.shape

torch.Size([10000, 128, 512])

In [ ]:
emb

tensor([[[ 0.0133, -0.0179, -0.0062,  ..., -0.0137,  0.0303, -0.1384],
         [ 0.0042, -0.0329,  0.0130,  ..., -0.0150,  0.0469, -0.0713],
         [-0.0136, -0.0014,  0.0171,  ..., -0.0329,  0.0310, -0.0701],
         ...,
         [-0.0039, -0.0673, -0.0077,  ...,  0.0545,  0.0871, -0.1126],
         [-0.0035, -0.0677, -0.0075,  ...,  0.0549,  0.0885, -0.1118],
         [-0.0034, -0.0682, -0.0072,  ...,  0.0550,  0.0890, -0.1112]],

        [[ 0.0262, -0.0136,  0.0048,  ..., -0.0134,  0.0425, -0.1287],
         [-0.0225, -0.0350,  0.0247,  ..., -0.0193,  0.0207, -0.0869],
         [ 0.0062, -0.0192,  0.0123,  ..., -0.0671,  0.0495, -0.0656],
         ...,
         [ 0.0048, -0.0677,  0.0028,  ...,  0.0575,  0.1026, -0.1017],
         [ 0.0046, -0.0686,  0.0019,  ...,  0.0582,  0.1044, -0.1016],
         [ 0.0058, -0.0660,  0.0025,  ...,  0.0547,  0.1010, -0.1029]],

        [[-0.0314, -0.0219, -0.0591,  ...,  0.0542,  0.0574, -0.1290],
         [-0.0277, -0.0095,  0.0205,  ..., -0

In [7]:
tokenizer(text)

{'input_ids': [[2, 18671, 17129, 17151, 17451, 17175, 17576, 21196, 17152, 17807, 21679, 20233, 24988, 18, 19509, 17491, 17389, 21196, 16, 18550, 17670, 17498, 22570, 9347, 16, 17670, 17498, 17516, 9339, 93, 17501, 153, 17718, 35, 18177, 3], [2, 17147, 17129, 17151, 17139, 20516, 17152, 17480, 17497, 18271, 17158, 17377, 16, 17134, 17348, 17176, 18623, 26593, 18, 3], [2, 20176, 17129, 17151, 17614, 18438, 17500, 17213, 23815, 20335, 8380, 3], [2, 20176, 17129, 18099, 17134, 18053, 17243, 17667, 17158, 17432, 23719, 9346, 21193, 41, 20478, 23489, 17155, 3], [2, 21051, 17129, 18560, 18118, 17178, 25130, 18748, 17191, 17134, 7, 17192, 28965, 16, 17151, 17211, 18229, 17393, 24396, 17160, 25640, 17277, 19642, 22858, 18, 17155, 3], [2, 19561, 17129, 17151, 18383, 17276, 26113, 17287, 69, 17161, 23855, 22800, 16, 17518, 17918, 17134, 18201, 17198, 16, 19472, 17371, 17157, 17174, 24189, 93, 6, 25076, 17448, 18600, 17301, 6, 24934, 22826, 19300, 24181, 18, 17816, 17918, 17134, 28978, 16, 17173,

In [13]:
tokens=tokenizer(text, return_tensors="pt", padding='max_length', max_length=128, truncation=True)
tokenizer.decode(tokens['input_ids'][0])

'[CLS] Jalisco _GEO _USR Se me hace raro que nunca salga nominada. Bueno ni tan raro, luego quien les lava, quien les cose y así ¿ verdad? jAjA [SEP] <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad> <pad>'

In [ ]:
tokenized_texts = [tokenizer.tokenize(t) for t in text]
tokenized_texts

[['Jalisco',
  '_GEO',
  '_USR',
  'Se',
  'me',
  'hace',
  'raro',
  'que',
  'nunca',
  'salga',
  'nom',
  '##inada',
  '.',
  'Bueno',
  'ni',
  'tan',
  'raro',
  ',',
  'luego',
  'quien',
  'les',
  'lav',
  '##a',
  ',',
  'quien',
  'les',
  'cos',
  '##e',
  'y',
  'así',
  '¿',
  'verdad',
  '?',
  'jAjA'],
 ['Mexico_City',
  '_GEO',
  '_USR',
  'Me',
  'cae',
  'que',
  'eso',
  'solo',
  'pasa',
  'en',
  'México',
  ',',
  'de',
  'todo',
  'se',
  'quieren',
  'aprovechar',
  '.'],
 ['Guerrero',
  '_GEO',
  '_USR',
  'Gracias',
  'hermosa',
  'siempre',
  'lo',
  'tendré',
  'presente',
  '🥰'],
 ['Guerrero',
  '_GEO',
  'Acaba',
  'de',
  'publicar',
  'una',
  'foto',
  'en',
  'Con',
  '##cepto',
  '##s',
  'Arte',
  'E',
  'Imp',
  '##resión',
  '_URL'],
 ['Coahuila',
  '_GEO',
  'Pres',
  '##entar',
  '##á',
  'alcalde',
  'elec',
  '##to',
  'de',
  '#',
  'Mon',
  '##clova',
  ',',
  '_USR',
  'su',
  'equipo',
  'cuando',
  'rin',
  '##da',
  'protesta',
  '

In [8]:
def generate_token_table_per_token(texts):
    embeddings, token_data = predict(texts)  # predict now returns tokens as well
    input_ids = token_data["input_ids"]
    attention_masks = token_data["attention_mask"]

    special_token_ids = set(tokenizer.all_special_ids + [30003, 2, 3])
    special_token_texts = {"[CLS]", "[SEP]", "<pad>"}


    data = []

    for i, tweet in enumerate(texts):
        region = tweet.split()[0]  # Asume que la región es la primera palabra del tweet
        ids = input_ids[i]
        masks = attention_masks[i]
        tokens = tokenizer.convert_ids_to_tokens(ids)

        for j, (token, token_id, mask) in enumerate(zip(tokens, ids, masks)):
            if j < embeddings.shape[1]:
                if mask == 0 or token_id.item() in special_token_ids or token in special_token_texts:
                    continue


                vector = embeddings[i, j].numpy()
                data.append({
                    "token": token,
                    "id_tweet": i,
                    "region": region,
                    "id_vector": j,
                    "vector": vector
                })
            else:
                print(f"Warning: Token index {j} out of bounds for embedding shape {embeddings.shape} at tweet {i}")

    df = pd.DataFrame(data)
    return df




In [9]:
df_tokens = generate_token_table_per_token(text)



In [ ]:
s

In [10]:
df_tokens

,token,id_tweet,region,id_vector,vector
0,Jalisco,0,Jalisco,1,"[0.0042095347, -0.032908052, 0.013027928, -0.0..."
1,_GEO,0,Jalisco,2,"[-0.013555932, -0.0013518701, 0.01710453, -0.0..."
2,_USR,0,Jalisco,3,"[-0.03582711, -0.044621233, 0.024303863, -0.04..."
3,Se,0,Jalisco,4,"[0.028649788, -0.02649775, 0.017882, -0.071087..."
4,me,0,Jalisco,5,"[-0.03112802, -0.05359524, 0.031188004, -0.074..."
...,...,...,...,...,...
296062,de,9999,NL,27,"[-0.01164437, -0.04061948, 0.05103537, -0.1034..."
296063,no,9999,NL,28,"[0.0319972, -0.057601426, 0.009991614, -0.1011..."
296064,se,9999,NL,29,"[0.00962469, -0.0075777457, 0.039374776, -0.11..."
296065,que,9999,NL,30,"[0.027089842, -0.05150798, 0.026053226, -0.041..."


In [11]:
df_tokens[df_tokens["id_tweet"]== 101]

,token,id_tweet,region,id_vector,vector
2962,Mexico_City,101,Mexico_City,1,"[-0.02840035, -0.0393182, 0.012395208, -0.0458..."
2963,_GEO,101,Mexico_City,2,"[-0.045251016, 0.006591415, -0.01348696, -0.05..."
2964,#,101,Mexico_City,3,"[9.005008e-05, -0.042418122, -0.009019574, -0...."
2965,art,101,Mexico_City,4,"[0.00013478418, -0.008461798, 0.061729435, -0...."
2966,##uro,101,Mexico_City,5,"[-0.004053139, 0.03296181, 0.01469787, -0.0515..."
2967,##ol,101,Mexico_City,6,"[0.041703433, 0.006368569, 0.15059933, -0.0478..."
2968,##ivar,101,Mexico_City,7,"[0.021034647, 0.034780208, 0.0045273337, -0.12..."
2969,##es,101,Mexico_City,8,"[0.00042054884, -0.025431741, 0.0006432506, -0..."
2970,#,101,Mexico_City,9,"[-0.0005869632, -0.036483128, -0.015184379, -0..."
2971,par,101,Mexico_City,10,"[0.008814191, -0.032448985, -0.003640226, -0.0..."
